In [ ]:
import json
import numpy as np
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer


# ============================================================
# Configuration
# ============================================================

# Input produced by the patient chunking pipeline
CHUNKS_PATH = Path("all_patient_chunks_v2.jsonl")

# Output that will contain the final patient-chunk embeddings
EMBEDDINGS_PATH = Path("patient_chunk_embeddings_v2.npy")

# Embedding model
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")

In [ ]:
# Load and validate the patient chunks.

# ============================================================
# Load patient chunks
# ============================================================

patient_chunks = []

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            patient_chunks.append(json.loads(line))

print(f"Total patient chunks loaded: {len(patient_chunks)}")

# Basic validation
required_fields = {
    "subject_id",
    "hadm_id",
    "section",
    "chunk_type",
    "text",
    "chunk_id"
}

missing_fields = []

for i, chunk in enumerate(patient_chunks):
    missing = required_fields - set(chunk.keys())
    if missing:
        missing_fields.append((i, missing))

if missing_fields:
    print(f"WARNING: {len(missing_fields)} chunks have missing fields.")
    print("First few:", missing_fields[:5])
else:
    print("All chunks contain the required fields.")

# Check for empty chunk text
empty_text = [
    i for i, chunk in enumerate(patient_chunks)
    if not str(chunk["text"]).strip()
]

print(f"Chunks with empty text: {len(empty_text)}")

# Number of unique admissions
unique_hadm_ids = {chunk["hadm_id"] for chunk in patient_chunks}
print(f"Unique admissions: {len(unique_hadm_ids)}")

In [ ]:
# Load BGE-base-en-v1.5.

# ============================================================
# Load embedding model
# ============================================================

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

print("Embedding model loaded successfully.")
print(f"Embedding dimension: {embedding_model.get_embedding_dimension()}")
print(f"Max sequence length: {embedding_model.max_seq_length}")

In [ ]:
# ============================================================
# Embed all patient chunks
# ============================================================

chunk_texts = [chunk["text"] for chunk in patient_chunks]

patient_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Number of embedded chunks: {len(patient_embeddings)}")
print(f"Embedding matrix shape: {patient_embeddings.shape}")
print(f"Embedding dtype: {patient_embeddings.dtype}")

In [ ]:
# Save and verify


# ============================================================
# Save patient chunk embeddings
# ============================================================

np.save(EMBEDDINGS_PATH, patient_embeddings)

print(f"Embeddings saved to: {EMBEDDINGS_PATH}")


# ============================================================
# Verify saved embeddings
# ============================================================

saved_embeddings = np.load(EMBEDDINGS_PATH)

assert saved_embeddings.shape == patient_embeddings.shape
assert len(saved_embeddings) == len(patient_chunks)

print(f"Saved embedding shape: {saved_embeddings.shape}")
print(f"Number of chunks: {len(patient_chunks)}")
print("Embedding/chunk alignment verified.")


# Show an example of the index-based connection
example_idx = 0

print("\nExample alignment:")
print(f"Index: {example_idx}")
print(f"Chunk ID: {patient_chunks[example_idx]['chunk_id']}")
print(f"HADM ID: {patient_chunks[example_idx]['hadm_id']}")
print(f"Section: {patient_chunks[example_idx]['section']}")
print(f"Text: {patient_chunks[example_idx]['text'][:300]}")
print(f"Embedding shape: {saved_embeddings[example_idx].shape}")